In [ ]:
import os
import inspect
from typing import override
import gradio as gr
from gradio.themes import Soft

from openai import OpenAI
from dotenv import load_dotenv

load_dotenv(override=True)

In [ ]:
OPENAI_API_KEY = os.environ.get("OPENAI_API_KEY")

if not OPENAI_API_KEY:
    raise RuntimeError(
        "Missing OPENAI_API_KEY. Set it in your environment or create a local .env file."
    )

client = OpenAI(api_key=OPENAI_API_KEY)

WHISPER_MODEL = os.environ.get("WHISPER_MODEL", "whisper-1")
LLM_MODEL = os.environ.get("LLM_MODEL", "gpt-4o-mini")

try:
    MAX_AUDIO_SIZE_MB = int(os.environ.get("MAX_AUDIO_SIZE_MB", "25"))
except ValueError:
    MAX_AUDIO_SIZE_MB = 25

try:
    LLM_MAX_OUTPUT_TOKENS = int(os.environ.get("LLM_MAX_OUTPUT_TOKENS", "5000"))
except ValueError:
    LLM_MAX_OUTPUT_TOKENS = 5000

SYSTEM_PROMPT = os.environ.get(
    "SYSTEM_PROMPT",
    """
You are a seasoned corporate documentation specialist and executive assistant,
expert in synthesizing meeting transcripts into formal minutes.

Your objectives:
- Perform a comprehensive analysis of the provided meeting transcript.
- Produce structured, professional meeting minutes in English.
- Use Markdown formatting (headers, bullet points, bold text) to ensure clarity
and professional presentation.
- Maintain high precision and conciseness by eliminating verbal fillers,
digressions, and non-essential dialogue.
- Explicitly identify and categorize action items, specifying owners
and deadlines.

Mandatory Response Structure:

## Executive Summary
[A concise 2-3 sentence overview of the meeting’s objective and primary
outcomes.]

## Participants
[List of attendees, if identified in the transcript.]

## Date and Venue
[Date, time, and location, if mentioned.]

## Agenda and Discussion Points
[Structured bullet points outlining the key topics and the progression
of the discussion.]

## Key Decisions and Strategic Conclusions
[Summary of the most significant decisions reached and formal conclusions.]

## Action Items
[Task list using the following format: **[OWNER]** — Description of task
— *Deadline*]

## Additional Notes
[Other relevant information, context, or follow-up requirements.]
""",
)


In [ ]:
def transcribe_audio(audio_file_path: str) -> str:
  """
  Sends an audio file to OpenAI Whisper and returns the transcription.

  This function processes a local audio file via the OpenAI Whisper API
  to convert speech into text. It includes a mandatory check for file
  size limits before transmission.

  Args:
    audio_file_path (str): The path to the audio file on the disk
    (e.g., "/tmp/meeting.mp3").

  Returns:
    str: The full transcription as a single string.

  Raises:
    ValueError: If the file size exceeds the 25MB limit.
    Exception: If there is an API error or a connection issue
    with OpenAI services.
  """

  file_size_mb = os.path.getsize(audio_file_path) / (1024 * 1024)

  if file_size_mb > MAX_AUDIO_SIZE_MB:
    raise ValueError(
        f"File is too large: {file_size_mb:.1f}MB."
        f"Maximum size is {MAX_AUDIO_SIZE_MB}MB."
    )

  print(f"Sending audio file ({file_size_mb:.1f}MB) to Whisper...")

  with open(audio_file_path, "rb") as audio_file:
    transcription = client.audio.transcriptions.create(
        model=WHISPER_MODEL,
        file=audio_file,
        response_format="text"
    )

    print(f"Transcription is ready! ({len(transcription)} chars).")
  
    return transcription

In [ ]:
def generate_meeting_minutes(transcription: str) -> str:
  """
  Processes a raw transcription using GPT-4o to generate structured meeting
  minutes.

  This function validates the input transcription, sends it to the OpenAI Chat
  Completions API using a specialized system prompt, and returns
  a professionally formatted summary in Markdown. It uses a low temperature
  setting (0.3) to prioritize factual accuracy and structural consistency
  over creativity.

  Args:
    transcription (str): The raw text transcript to be analyzed.

  Returns:
    str: The finalized meeting minutes formatted in Markdown, containing
    summaries, action items, and key decisions.

  Raises:
    ValueError: If the provided transcription is empty or contains only
    whitespace.
    Exception: If the OpenAI API request fails due to connectivity,
    authentication, or rate-limiting issues.
  """

  if not transcription or not transcription.strip():
    raise ValueError("Transcription is empty - there is nothing to analyze.")

  print(f"Sending transcription ({len(transcription)} chars) to GPT...")

  response = client.chat.completions.create(
      model=LLM_MODEL,
      messages=[
          {
              "role": "system",
              "content": SYSTEM_PROMPT
          },
          {
              "role": "user",
              "content": f"""
              Below is a transcript of the meeting.
              Generate professional minutes according to the provided format.

              TRANSCRIPTION:
              ---
              {transcription}
              ---
              """
          }
      ],
      max_tokens=LLM_MAX_OUTPUT_TOKENS,
      temperature=0.3
  )

  minutes = response.choices[0].message.content
  
  if minutes is None:
    raise ValueError("OpenAI returned no message content for meeting minutes.")

  print(f"Minutes generated! ({len(minutes)} chars).")

  return minutes

In [ ]:
def process_meeting_audio(
    audio_file,
    model: str,
    meeting_context: str,
) -> tuple[str, str]:
  """End-to-end: transcription -> minutes (English-only)."""

  if audio_file is None:
    return "No audio file provided.", "No audio file provided."

  try:
    global LLM_MODEL

    LLM_MODEL = model or globals().get("LLM_MODEL", "gpt-4o-mini")

    transcription = transcribe_audio(audio_file)

    if meeting_context and meeting_context.strip():
      transcription_for_minutes = (
          f"MEETING CONTEXT:\n---\n{meeting_context.strip()}\n---\n\n" + transcription
      )
    else:
      transcription_for_minutes = transcription

    minutes = generate_meeting_minutes(transcription_for_minutes)
    
    return transcription, minutes

  except ValueError as e:
    error_msg = f"Validation Error: {str(e)}"
    return error_msg, error_msg
  except Exception as e:
    error_msg = f"An error occurred: {str(e)}"
    return error_msg, error_msg

In [ ]:
textbox_kwargs = {}
if "show_copy_button" in inspect.signature(gr.Textbox.__init__).parameters:
  textbox_kwargs["show_copy_button"] = True


def _audio_file_info_md(audio_path: str | None) -> str:
  if not audio_path:
    return "**Current audio:** _No file selected._"

  try:
    size_mb = os.path.getsize(audio_path) / (1024 * 1024)
  except Exception:
    size_mb = None

  filename = os.path.basename(audio_path)

  if size_mb is None:
    return f"**Current audio:** `{filename}`"

  return f"**Current audio:** `{filename}` ({size_mb:.1f} MB)"


def _safe_html(text: str) -> str:
  return (text or "").replace("&", "&amp;").replace("<", "&lt;").replace(">", "&gt;")


def _loader_html() -> str:
  return (
      "<div class=\"mm-loader\" aria-label=\"Loading\">"
      "<span class=\"mm-spinner\" aria-hidden=\"true\"></span>"
      "</div>"
  )


def _reset_outputs():
  return (
      gr.update(value=""),
      gr.update(value="*Minutes will appear here after generating...*"),
  )


def on_audio_change(audio_path: str | None):
  current_audio_md = _audio_file_info_md(audio_path)
  transcription_reset, minutes_reset = _reset_outputs()

  if not audio_path:
    return (
        gr.update(value=current_audio_md),
        gr.update(interactive=False),
        transcription_reset,
        minutes_reset,
    )

  return (
      gr.update(value=current_audio_md),
      gr.update(interactive=True),
      gr.update(),
      gr.update(),
  )


def _stream_minutes(transcription_for_minutes: str):
  resp = client.chat.completions.create(
      model=LLM_MODEL,
      messages=[
          {"role": "system", "content": SYSTEM_PROMPT},
          {
              "role": "user",
              "content": f"""
              Below is a transcript of the meeting.
              Generate professional minutes according to the provided format.

              TRANSCRIPTION:
              ---
              {transcription_for_minutes}
              ---
              """,
          },
      ],
      max_tokens=LLM_MAX_OUTPUT_TOKENS,
      temperature=0.3,
      stream=True,
  )

  acc = ""

  for chunk in resp:
    try:
      delta = chunk.choices[0].delta
      piece = getattr(delta, "content", None)
    except Exception:
      piece = None

    if piece:
      acc += piece
      yield acc


def process_meeting_audio_ui(audio_file, model: str, meeting_context: str):
  if audio_file is None:
    return (
        gr.update(value=""),
        gr.update(value="*Minutes will appear here after generating...*"),
        gr.update(value="", visible=False),
        gr.update(value="", visible=False),
        gr.update(interactive=True),
        gr.update(interactive=True),
        gr.update(interactive=True),
        gr.update(interactive=False),
    )

  try:
    yield (
        gr.update(value=""),
        gr.update(),
        gr.update(value=_loader_html(), visible=True),
        gr.update(value=_loader_html(), visible=True),
        gr.update(interactive=False),  
        gr.update(interactive=False),  
        gr.update(interactive=False),  
        gr.update(interactive=False), 
    )

    global LLM_MODEL

    LLM_MODEL = model or globals().get("LLM_MODEL", "gpt-4o-mini")

    transcription = transcribe_audio(audio_file)

    yield (
        gr.update(value=transcription),
        gr.update(),
        gr.update(value="", visible=False),
        gr.update(value=_loader_html(), visible=True),
        gr.update(interactive=False),
        gr.update(interactive=False),
        gr.update(interactive=False),
        gr.update(interactive=False),
    )

    if meeting_context and meeting_context.strip():
      transcription_for_minutes = (
          f"MEETING CONTEXT:\n---\n{meeting_context.strip()}\n---\n\n" + transcription
      )
    else:
      transcription_for_minutes = transcription

    minutes_acc = ""

    for minutes_acc in _stream_minutes(transcription_for_minutes):
      yield (
          gr.update(value=transcription),
          gr.update(value=minutes_acc),
          gr.update(value="", visible=False),
          gr.update(value=_loader_html(), visible=True),
          gr.update(interactive=False),
          gr.update(interactive=False),
          gr.update(interactive=False),
          gr.update(interactive=False),
      )

    yield (
        gr.update(value=transcription),
        gr.update(value=minutes_acc),
        gr.update(value="", visible=False),
        gr.update(value="", visible=False),
        gr.update(interactive=True),
        gr.update(interactive=True),
        gr.update(interactive=True),
        gr.update(interactive=True),
    )

  except ValueError as e:
    msg = f"Validation error: {str(e)}"

    yield (
        gr.update(value=msg),
        gr.update(value=msg),
        gr.update(value="", visible=False),
        gr.update(value="", visible=False),
        gr.update(interactive=True),
        gr.update(interactive=True),
        gr.update(interactive=True),
        gr.update(interactive=True),
    )
  except Exception as e:
    msg = f"Error: {str(e)}"

    yield (
        gr.update(value=msg),
        gr.update(value=msg),
        gr.update(value="", visible=False),
        gr.update(value="", visible=False),
        gr.update(interactive=True),
        gr.update(interactive=True),
        gr.update(interactive=True),
        gr.update(interactive=True),
    )


with gr.Blocks(title="Meeting Minutes AI") as demo:
  gr.Markdown(
      """
      # Meeting Minutes AI
      Upload meeting recording -> receive professional notes automatically.

      **Supported formats:** MP3, WAV, M4A, OGG, FLAC, WEBM.
      **Size limit:** 25MB.
      """
  )

  audio_input = gr.Audio(
      label="Upload audio file",
      type="filepath",
      sources=["upload"],
  )

  current_audio_md = gr.Markdown(_audio_file_info_md(None))

  model_input = gr.Textbox(
      label="LLM model",
      value=LLM_MODEL,
      placeholder="e.g. gpt-4o-mini",
  )

  meeting_context_input = gr.Textbox(
      label="Meeting context (optional)",
      lines=4,
      placeholder="e.g. project name, expected attendees, goal of the meeting...",
  )

  submit_btn = gr.Button("Generate minutes", variant="primary", size="lg", interactive=False)

  gr.Markdown(
      """
      **How to use:**
      1. Upload an audio file.
      2. Confirm the correct file is selected.
      3. Click "Generate minutes".
      4. Wait ~30-60 seconds.
      5. Copy the results.
      """
  )

  with gr.Tabs():
    with gr.Tab("Minutes"):
      minutes_output = gr.Markdown(
          label="Meeting minutes",
          value="*Minutes will appear here after generating...*",
          elem_id="mm-minutes-output",
      )

      minutes_loader = gr.HTML(value="", visible=False, elem_id="mm-minutes-loader")

    with gr.Tab("Transcription"):
      transcription_output = gr.Textbox(
          label="Raw transcription",
          lines=15,
          placeholder="The raw transcript will appear here…",
          elem_id="mm-transcription-output",
          **textbox_kwargs,
      )
      
      transcription_loader = gr.HTML(value="", visible=False, elem_id="mm-transcription-loader")

  audio_input.change(
      fn=on_audio_change,
      inputs=[audio_input],
      outputs=[
          current_audio_md,
          submit_btn,
          transcription_output,
          minutes_output,
      ],
  )

  submit_btn.click(
      fn=process_meeting_audio_ui,
      inputs=[audio_input, model_input, meeting_context_input],
      outputs=[
          transcription_output,
          minutes_output,
          transcription_loader,
          minutes_loader,
          audio_input,
          model_input,
          meeting_context_input,
          submit_btn,
      ],
  )

In [ ]:
demo.launch(
    share=False,
    debug=True,
    theme=Soft(),
    css="""
    .header-text { text-align: center; margin-bottom: 20px; }
    .output-box { font-family: 'Courier New', monospace; }

    .mm-loader {
      display: flex;
      justify-content: center;
      padding: 10px 0 0;
    }

    .mm-spinner {
      width: 16px;
      height: 16px;
      border-radius: 50%;
      border: 2px solid rgba(255, 255, 255, 0.25);
      border-top-color: rgba(255, 255, 255, 0.85);
      animation: mmspin 0.9s linear infinite;
    }

    @keyframes mmspin {
      to { transform: rotate(360deg); }
    }

    #mm-minutes-output,
    #mm-minutes-output * {
      border: none !important;
      box-shadow: none !important;
      outline: none !important;
      background: transparent !important;
    }

    #mm-minutes-output::before,
    #mm-minutes-output::after,
    #mm-minutes-output *::before,
    #mm-minutes-output *::after {
      border: none !important;
      box-shadow: none !important;
      background: transparent !important;
    }

    #mm-minutes-output .prose {
      padding-top: 6px;
    }

    /* Do not use #mm-minutes-loader * — it strips .mm-spinner borders. */
    #mm-minutes-loader,
    #mm-transcription-loader {
      border: none !important;
      box-shadow: none !important;
      outline: none !important;
      background: transparent !important;
    }
    """,
)